In [0]:
       
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Prepare Orders Data
def prepare_orders(fact_orders, fact_order_items):
    """
    Joins orders and items, extracting the flags needed for operational metrics.
    """
    return fact_orders.alias("o").join(
        fact_order_items.alias("oi"), 
        "order_id", 
        "inner"
    ).select(
        "oi.seller_key",
        "o.order_id",
        F.to_date("o.order_purchase_timestamp").alias("order_date"),
        "o.late_delivery_flag",
        "o.order_status"
    ).dropDuplicates()

# 2. Prepare Reviews

def prepare_pit_reviews(fact_reviews):
    """
    Creates a historical ledger of reviews to prevent data leakage.
    Each review state gets a 'valid_from' and 'valid_to' date.
    """
    # Get the very first review date per order (this anchors the 60-day window limit)
    first_dates = fact_reviews.groupBy("seller_key", "order_id").agg(
        F.min(F.to_date("review_creation_date")).alias("first_review_date")
    )
    
    # Create effective date ranges for each score update 
    w_history = Window.partitionBy("seller_key", "order_id").orderBy("review_creation_date")
    
    pit_reviews = fact_reviews.join(first_dates, ["seller_key", "order_id"], "inner") \
        .withColumn("valid_from", F.to_date("review_creation_date")) \
        .withColumn("valid_to", F.coalesce(
            F.date_sub(F.to_date(F.lead("review_creation_date").over(w_history)), 1),
            F.to_date(F.lit("2099-12-31")) # Currently active state
        )) \
        .withColumn("is_claim", F.when(F.col("review_score") < 3, 1).otherwise(0))
        
    return pit_reviews

# 3. Create Dense Calendar with Lookback Windows
def create_dense_calendar(df_orders, df_items):
    """
    Generates the base matrix of Seller x Date, including
    60-day lookback boundaries for each reference date.
    """
    sellers = df_items.select("seller_key").distinct()
    
    date_bounds = df_orders.select(
        F.min("order_date").alias("min_date"), 
        F.max("order_date").alias("max_date")
    ).collect()[0]
    
    calendar = spark.sql(f"""
        SELECT EXPLODE(SEQUENCE(
            DATE '{date_bounds["min_date"]}', 
            DATE '{date_bounds["max_date"]}', 
            INTERVAL 1 DAY
        )) AS reference_date
    """)
    
    # Cross join and define the 60-day window
    return sellers.crossJoin(calendar) \
        .withColumn("window_start", F.date_sub("reference_date", 62)) \
        .withColumn("window_end", F.date_sub("reference_date", 2))

# 4. Point-in-Time Aggregation
def calculate_pit_metrics(calendar, pit_orders, pit_reviews):
    """
    Joins events strictly within the 60-day window.
    """
    
    # 4.1 Calculate Order Metrics 
    # Only orders that happened within the [ref - 62, ref - 2] window
    order_metrics = calendar.alias("c").join(
        pit_orders.alias("o"),
        (F.col("o.seller_key") == F.col("c.seller_key")) &
        (F.col("o.order_date").between(F.col("c.window_start"), F.col("c.window_end"))),
        "left"
    ).groupBy("c.seller_key", "c.reference_date").agg(
        F.count("o.order_id").alias("orders_60d"),
        F.sum(F.when(F.col("o.late_delivery_flag") == 1, 1).otherwise(0)).alias("late_shipments_60d"),
        F.sum(F.when(F.col("o.order_status") == 'canceled', 1).otherwise(0)).alias("cancellations_60d")
    )
    
    # 4.2 Calculate Review Metrics
    
    # The original review must have been created within the the [ref - 62, ref - 2] window 
  
    review_metrics = calendar.alias("c").join(
        pit_reviews.alias("r"),
        (F.col("r.seller_key") == F.col("c.seller_key")) &
        (F.col("r.first_review_date").between(F.col("c.window_start"), F.col("c.window_end"))) &
        (F.col("c.reference_date").between(F.col("r.valid_from"), F.col("r.valid_to"))),
        "left"
    ).groupBy("c.seller_key", "c.reference_date").agg(
        F.count("r.order_id").alias("reviews_60d"),
        F.sum("r.is_claim").alias("claims_60d"),
        F.sum("r.review_score").alias("sum_score_60d")
    )
    
    # 4.3 Combine and calculate final rates
    combined = order_metrics.join(review_metrics, ["seller_key", "reference_date"], "inner")
    
    # Avoiding inconsistencies when dividing by 0
    def safe_div(numerator, denominator):
        return F.col(numerator) / F.when(F.col(denominator) == 0, F.lit(None)).otherwise(F.col(denominator))

    df_final = combined.withColumn("avg_review_score_60d", F.round(safe_div("sum_score_60d", "reviews_60d"), 2)) \
                       .withColumn("late_rate", F.round(safe_div("late_shipments_60d", "orders_60d"), 2)) \
                       .withColumn("cancel_rate", F.round(safe_div("cancellations_60d", "orders_60d"), 2)) \
                       .withColumn("claim_rate", F.round(safe_div("claims_60d", "reviews_60d"), 2)) \
                       .withColumn("reliability_orders", F.round(F.col("orders_60d") / (F.col("orders_60d") + 20), 2)) \
                       .withColumn("reliability_reviews", F.round(F.col("reviews_60d") / (F.col("reviews_60d") + 20), 2)) \
                       .drop("sum_score_60d") \
                       .sort( F.asc("seller_key"), F.asc("reference_date")) # New for testing

    # Keep only active sellers based on the rules
    return df_final.filter((F.col("orders_60d") > 0) | (F.col("reviews_60d") > 0))

# PIPELINE EXECUTION

# 1. Read source tables
fact_orders = spark.read.table("marketplace_olist.gold.fact_orders")
fact_order_items = spark.read.table("marketplace_olist.gold.fact_order_items")
fact_reviews = spark.read.table("marketplace_olist.gold.fact_reviews")

# 2. Prepare Data and State Ledgers
pit_orders = prepare_orders(fact_orders, fact_order_items)
pit_reviews = prepare_pit_reviews(fact_reviews)
dense_calendar = create_dense_calendar(pit_orders, fact_order_items)

# 3. Calculate final metrics dynamically
df_fact_seller_metrics = calculate_pit_metrics(dense_calendar, pit_orders, pit_reviews)


In [0]:
# Save to Gold Medallion Layer
df_fact_seller_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("marketplace_olist.gold.fact_seller_metrics")